In [1]:
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW = os.path.join(PROJECT_ROOT, "data/raw")
PROC = os.path.join(PROJECT_ROOT, "data/processed")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
print("Project root:", PROJECT_ROOT)

Project root: f:\BRACU\CSE425\Project\gnn-bert-music-context


In [ ]:
# ============================================================
# CELL 1 — Project root + imports
# ============================================================
import os, glob, json
import pandas as pd
import numpy as np
import torch
import librosa
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from torch_geometric.data import Data

d:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: f:\BRACU\CSE425\Project\gnn-bert-music-context


In [ ]:
# ============================================================
# CELL 2 — Graph-building functions 
# ============================================================
def build_segment_graph(segment_features, top_k=1):
    n = len(segment_features)
    edges = []
    for i in range(n - 1):
        edges.append((i, i + 1))
        edges.append((i + 1, i))
    sim_matrix = cosine_similarity(segment_features)
    for i in range(n):
        candidates = [(j, sim_matrix[i, j]) for j in range(n) if abs(j - i) > 1]
        candidates.sort(key=lambda x: -x[1])
        for j, score in candidates[:top_k]:
            if (i, j) not in edges:
                edges.append((i, j))
                edges.append((j, i))
    return edges, sim_matrix

def to_pyg_data(segment_features, edges):
    x = torch.tensor(segment_features, dtype=torch.float)
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

def process_track(audio_path, sr=22050, segment_sec=5, top_k=1):
    y, _ = librosa.load(audio_path, sr=sr)
    segment_samples = segment_sec * sr
    n_segments = len(y) // segment_samples
    if n_segments < 2:
        return None
    segment_features = []
    for i in range(n_segments):
        start = i * segment_samples
        end = start + segment_samples
        seg = y[start:end]
        chroma = librosa.feature.chroma_stft(y=seg, sr=sr, n_chroma=12)
        segment_features.append(chroma.mean(axis=1))
    segment_features = np.array(segment_features)
    edges, _ = build_segment_graph(segment_features, top_k=top_k)
    return to_pyg_data(segment_features, edges)

In [ ]:
# ============================================================
# CELL 3 — 2a: FMA -> chroma segment graphs 
# ============================================================
FMA_DIR = os.path.join(RAW, "fma")
GRAPH_DIR = os.path.join(PROC, "fma_graphs")
os.makedirs(GRAPH_DIR, exist_ok=True)

FMA_MANIFEST_PATH = os.path.join(PROC, "fma_graph_manifest.csv")
if os.path.exists(FMA_MANIFEST_PATH):
    fma_manifest = pd.read_csv(FMA_MANIFEST_PATH)
else:
    fma_manifest = pd.DataFrame(columns=["track_id", "success", "error"])

done_track_ids = set(fma_manifest["track_id"])  
print(f"Already attempted: {len(done_track_ids)}")

all_mp3s = glob.glob(os.path.join(FMA_DIR, "fma_small", "**", "*.mp3"), recursive=True)
print("Total mp3 files found:", len(all_mp3s))  

Already attempted: 0
Total mp3 files found: 8000


In [ ]:
# ============================================================
# CELL 4 — 2a: batch graph-building 
# ============================================================
new_rows = []
for mp3_path in tqdm(all_mp3s):
    track_id = int(os.path.basename(mp3_path).replace(".mp3", ""))
    if track_id in done_track_ids:
        continue
    try:
        graph = process_track(mp3_path, segment_sec=5, top_k=1)
        if graph is None:
            new_rows.append({"track_id": track_id, "success": False, "error": "too short"})
        else:
            torch.save(graph, os.path.join(GRAPH_DIR, f"{track_id}.pt"))
            new_rows.append({"track_id": track_id, "success": True, "error": ""})
    except Exception as e:
        new_rows.append({"track_id": track_id, "success": False, "error": str(e)[:200]})
    done_track_ids.add(track_id)

fma_manifest = pd.concat([fma_manifest, pd.DataFrame(new_rows)]).drop_duplicates(subset="track_id", keep="last")
fma_manifest.to_csv(FMA_MANIFEST_PATH, index=False)
print(fma_manifest["success"].value_counts())

  9%|▉         | 737/8000 [03:14<18:57,  6.38it/s]  d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
  9%|▉         | 738/8000 [03:14<18:38,  6.49it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 12%|█▏        | 972/8000 [03:56<15:27,  7.57it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 15%|█▍        | 1191/8000 [04:29<12:50,  8.83it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 19%|█▊        | 1494/8000 [05:12<19:53,  5.45it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty

success
True     7994
False       6
Name: count, dtype: int64


In [5]:
# ============================================================
# CELL 5 — 2a: merge with genre labels
# ============================================================
fma_labels = pd.read_csv(os.path.join(PROC, "fma_small_labels.csv"))
successful_ids = set(fma_manifest[fma_manifest["success"] == True]["track_id"])
fma_labels_final = fma_labels[fma_labels["track_id"].isin(successful_ids)]
fma_labels_final.to_csv(os.path.join(PROC, "fma_small_labels_final.csv"), index=False)
print(fma_labels_final.shape)
print(fma_labels_final["genre"].value_counts())

(7994, 2)
genre
Pop              1000
Folk             1000
International    1000
Instrumental     1000
Experimental      999
Rock              999
Electronic        999
Hip-Hop           997
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 6 — 2b: MusicCaps -> chroma graphs 
# ============================================================
MC_DIR = os.path.join(RAW, "musiccaps")
MC_AUDIO_DIR = os.path.join(MC_DIR, "audio")
MC_GRAPH_DIR = os.path.join(PROC, "musiccaps_graphs")
os.makedirs(MC_GRAPH_DIR, exist_ok=True)

MC_MANIFEST_PATH = os.path.join(PROC, "musiccaps_graph_manifest.csv")
if os.path.exists(MC_MANIFEST_PATH):
    mc_graph_manifest = pd.read_csv(MC_MANIFEST_PATH)
else:
    mc_graph_manifest = pd.DataFrame(columns=["ytid", "success", "error"])

done_ytids = set(mc_graph_manifest["ytid"])
all_wavs = glob.glob(os.path.join(MC_AUDIO_DIR, "*.wav"))
print("Total wav files:", len(all_wavs))  

new_rows = []
for wav_path in tqdm(all_wavs):
    ytid = os.path.basename(wav_path).replace(".wav", "")
    if ytid in done_ytids:
        continue
    try:
        graph = process_track(wav_path, segment_sec=2, top_k=1)
        if graph is None:
            new_rows.append({"ytid": ytid, "success": False, "error": "too short"})
        else:
            torch.save(graph, os.path.join(MC_GRAPH_DIR, f"{ytid}.pt"))
            new_rows.append({"ytid": ytid, "success": True, "error": ""})
    except Exception as e:
        new_rows.append({"ytid": ytid, "success": False, "error": str(e)[:200]})
    done_ytids.add(ytid)

mc_graph_manifest = pd.concat([mc_graph_manifest, pd.DataFrame(new_rows)]).drop_duplicates(subset="ytid", keep="last")
mc_graph_manifest.to_csv(MC_MANIFEST_PATH, index=False)
print(mc_graph_manifest["success"].value_counts())

Total wav files: 1159


  3%|▎         | 39/1159 [00:02<00:51, 21.72it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 11%|█         | 130/1159 [00:06<00:47, 21.84it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 18%|█▊        | 205/1159 [00:10<00:44, 21.42it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 33%|███▎      | 379/1159 [00:18<00:35, 22.04it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
 34%|███▍      | 394/1159 [00:18<00:33, 22.79it/s]d:\Program Files\Python\Lib\site-packages\librosa\core\pitch.py:105: UserWarning: Trying to estimate tuning from empty freq

success
True    1159
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 7 — 2b: BERT tokenization of MusicCaps captions
# ============================================================
from transformers import BertTokenizer

mc = pd.read_csv(os.path.join(MC_DIR, "musiccaps-public.csv"))
successful_mc_ids = set(mc_graph_manifest[mc_graph_manifest["success"] == True]["ytid"])
mc_final = mc[mc["ytid"].isin(successful_mc_ids)].reset_index(drop=True)
print("Matched rows:", len(mc_final))

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
all_tokens = tokenizer(
    mc_final["caption"].tolist(),
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
)

MC_TEXT_DIR = os.path.join(PROC, "musiccaps_text")
os.makedirs(MC_TEXT_DIR, exist_ok=True)
torch.save({
    "input_ids": all_tokens["input_ids"],
    "attention_mask": all_tokens["attention_mask"],
    "ytids": mc_final["ytid"].tolist(),
}, os.path.join(MC_TEXT_DIR, "musiccaps_tokenized.pt"))

print(all_tokens["input_ids"].shape, all_tokens["attention_mask"].shape)

Matched rows: 1159


d:\Program Files\Python\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zawad\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


torch.Size([1159, 128]) torch.Size([1159, 128])


In [ ]:
# ============================================================
# CELL 8 — 2c: MagnaTagATune confirmation 
# ============================================================
mtat_top50 = pd.read_csv(os.path.join(PROC, "mtat_top50_tags.csv"))
print(mtat_top50.shape) 

(25863, 52)
